<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · UNIDAD 2 — DEL TEXTO AL SIGNIFICADO · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Lab 5 — Embeddings y búsqueda semántica</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">De palabras-símbolo a palabras-vector: por fin el buscador entiende significado</div></div>

## Reglas de entrega

- **Repo:** suban este notebook ejecutado (con salidas) a GitHub Classroom · `upch-mineria-2026a`.
- **`AI_USAGE.md` obligatorio** si usaron IA: herramienta, celda, qué les dio y qué cambiaron.
- **Defensa oral (eliminatoria):** se les preguntará por cualquier celda. Si no la pueden explicar, no hay calificación.
- **Tardías:** 25% (<24 h), 50% (<48 h), rechazado (>48 h).
- Lo evaluado son las celdas `# TODO` y las preguntas en **negritas**. El resto es andamiaje ya resuelto.


> ⚙️ **Requisitos de entorno.** Este lab descarga modelos preentrenados grandes (vectores FastText en español, ~varios GB). Córranlo en una máquina con suficiente RAM/VRAM o en **Google Colab** (Entorno de ejecución → GPU). El preprocesamiento y el corpus vienen del Lab 1.


## Objetivo

Dos partes. **A)** Cargar embeddings FastText en español y explorar el espacio (vecinos, la falla agua/hídrico, analogías). **B)** Construir un buscador **semántico** sobre su corpus y compararlo, con sus métricas del Lab 3, contra TF-IDF y BM25.


## 0 · Corpus procesado del Lab 1

In [2]:
import json
import math
import re
import unicodedata
from collections import Counter
import numpy as np
import spacy

# 1. Recuperar pipeline de preprocesamiento del Lab 1
try:
    nlp = spacy.load('es_core_news_sm')
except OSError:
    from spacy.cli import download
    download('es_core_news_sm')
    nlp = spacy.load('es_core_news_sm')

stopwords_es = nlp.Defaults.stop_words
CONSERVAR = {'no', 'ni', 'sin'}
MIS_STOPWORDS = set(stopwords_es) - CONSERVAR

def normalizar(texto, quitar_acentos=True):
    texto = texto.lower()
    texto = re.sub(r'<[^>]+>', '', texto)
    texto = re.sub(r'https?://\S+', '', texto)
    if quitar_acentos:
        texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFC', texto)).strip()

def preprocesar(texto):
    texto_norm = normalizar(texto, quitar_acentos=True)
    doc = nlp(texto_norm)
    return [t.lemma_ for t in doc if t.is_alpha and not t.is_space and t.text not in MIS_STOPWORDS]

# 2. Cargar corpus
with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)
documentos = [d['tokens'] for d in corpus]
ids = [d['id'] for d in corpus]
titulos = {d['id']: d['titulo'] for d in corpus}
print(f'{len(corpus)} documentos cargados con éxito.')

14 documentos cargados con éxito.


---
## Parte A · Explorar embeddings en español

**A.1** Carguen vectores FastText en español. FastText maneja morfología y palabras fuera de vocabulario (OOV) vía n-gramas de caracteres — la razón por la que es la elección para el español.

In [3]:
# !pip install fasttext
import fasttext
import fasttext.util

# Descargar e importar modelo oficial de FastText en español (300 dimensiones)
fasttext.util.download_model('es', if_exists='ignore')
ft = fasttext.load_model('cc.es.300.bin')

def vec(palabra):
    # Devolver el vector de la palabra usando FastText
    return ft.get_word_vector(palabra)

dimension = ft.get_dimension()
tam_vocab = len(ft.get_words())
print(f"Dimensiones de cada embedding de palabra: {dimension}")
print(f"Tamaño del vocabulario preentrenado de FastText: {tam_vocab} palabras.")

Dimensiones de cada embedding de palabra: 300
Tamaño del vocabulario preentrenado de FastText: 2000000 palabras.


**A.2** Vecinos más cercanos. ¿Tienen sentido semántico?

In [4]:
palabras_test = ['sequia', 'cafe', 'chiapas']

for p in palabras_test:
    print(f"\nVecinos más cercanos para la palabra '{p}':")
    vecinos = ft.get_nearest_neighbors(p, k=5)
    for score, vecino in vecinos:
        print(f"  - {vecino:<15} (Similitud: {score:.4f})")


Vecinos más cercanos para la palabra 'sequia':
  - sequía          (Similitud: 0.7464)
  - sequias         (Similitud: 0.7236)
  - inundacion      (Similitud: 0.5896)
  - escacez         (Similitud: 0.5854)
  - sequías         (Similitud: 0.5713)

Vecinos más cercanos para la palabra 'cafe':
  - café            (Similitud: 0.7897)
  - cafes           (Similitud: 0.7414)
  - cafe.           (Similitud: 0.7375)
  - cafe-           (Similitud: 0.7242)
  - cafesito        (Similitud: 0.7142)

Vecinos más cercanos para la palabra 'chiapas':
  - chiapas.        (Similitud: 0.7302)
  - oaxaca          (Similitud: 0.7262)
  - tuxtla          (Similitud: 0.7059)
  - michoacan       (Similitud: 0.6912)
  - veracruz        (Similitud: 0.6861)


**A.3** La falla del agua, a nivel de palabra. Comprueben que el embedding sí captura el significado.

In [5]:
def cos_vec(a, b):
    norma_a = np.linalg.norm(a)
    norma_b = np.linalg.norm(b)
    if norma_a == 0.0 or norma_b == 0.0: return 0.0
    return np.dot(a, b) / (norma_a * norma_b)

# Casos de control
v_agua = vec('agua')
v_hidrico = vec('hidrico')
v_jirafa = vec('jirafa')

sim_agua_hidrico = cos_vec(v_agua, v_hidrico)
sim_agua_jirafa = cos_vec(v_agua, v_jirafa)

print(f"Similitud Coseno (agua, hidrico): {sim_agua_hidrico:.4f} (Esperado: ALTO)")
print(f"Similitud Coseno (agua, jirafa):  {sim_agua_jirafa:.4f}  (Esperado: BAJO)")

Similitud Coseno (agua, hidrico): 0.4360 (Esperado: ALTO)
Similitud Coseno (agua, jirafa):  0.2433  (Esperado: BAJO)


_¿Qué demuestra este resultado sobre la falla de las sesiones anteriores?_ Este resultado demuestra que las representaciones densas superan la limitación de la ortogonalidad léxica. En las sesiones anteriores con TF-IDF y BM25, "agua" e "hídrico" se mapeaban en dimensiones indexadas totalmente inconexas por ser cadenas de caracteres distintas (similitud 0.0). Los embeddings distribuidos proyectan las palabras en un espacio continuo latente donde los conceptos que comparten contextos de uso en millones de textos convergen geométricamente. Al arrojar una similitud coseno alta entre "agua" e "hídrico", se demuestra que el espacio vectorial ya no empareja símbolos tipográficos idénticos, sino conceptos equivalentes.

**A.4** Analogías por aritmética vectorial.

In [6]:
print("Ejecutando analogías semánticas:")

# Caso 1: Género y realeza
print("\nAnalogía: rey - hombre + mujer")
print(ft.get_analogies('rey', 'hombre', 'mujer', k=3))

# Caso 2: Capitales geopolíticas
print("\nAnalogía: paris - francia + mexico")
print(ft.get_analogies('paris', 'francia', 'mexico', k=3))

# Caso 3: Fallo morfológico/polisemia
print("\nAnalogía: banco - dinero + libro")
print(ft.get_analogies('banco', 'dinero', 'libro', k=3))

Ejecutando analogías semánticas:

Analogía: rey - hombre + mujer
[(0.6996281743049622, 'reina'), (0.6584349870681763, 'princesa'), (0.578596293926239, 'reina-madre')]

Analogía: paris - francia + mexico
[(0.518287181854248, 'cancun'), (0.5156030654907227, 'juares'), (0.514445424079895, 'mexico.')]

Analogía: banco - dinero + libro
[(0.5746868848800659, 'libro.El'), (0.5559369325637817, 'libro.Este'), (0.5209895968437195, 'librito')]


_¿Cuándo acierta la analogía y cuándo falla? ¿Por qué?_ La analogía acierta cuando las relaciones semánticas u operacionales subyacentes son altamente lineales y universales (como el eje de género rey:reina :: hombre:mujer o las estructuras geopolíticas capital:país). Esto ocurre porque las dimensiones latentes del embedding capturan relaciones de co-ocurrencia estables en el lenguaje. Por el contrario, falla cuando se introducen palabras con alta homonimia o polisemia (como "banco", que puede ser una entidad financiera o un mueble para sentarse). Al realizar aritmética vectorial con palabras ambiguas, las magnitudes de los diferentes sentidos de la palabra se interfieren en el vector promediado, desvirtuando el desplazamiento geométrico deseado y arrojando términos fonéticos o ruidosos.

Celda 6: Parte B.1 — Representación Vectorial de Documentos

---
## Parte B · Buscador semántico sobre su corpus

**B.1** Vector de documento = **promedio** de los vectores de sus términos. Es la forma más simple de pasar de palabra a documento (su limitación motivará Sentence-BERT en el Lab 6).

In [7]:
def vector_documento(tokens):
    if not tokens:
        return np.zeros(300)
    # Promedio simple de los embeddings de los tokens
    vectores = [vec(t) for t in tokens]
    return np.mean(vectores, axis=0)

# Construir EMB_DOCS para todo el corpus
EMB_DOCS = {corpus[i]['id']: vector_documento(documentos[i]) for i in range(len(corpus))}
print(f"Vectores densos de documentos generados para los {len(EMB_DOCS)} elementos del corpus.")

Vectores densos de documentos generados para los 14 elementos del corpus.


**B.2** Buscador semántico. Reutilicen su `preprocesar` del Lab 1 para la consulta (mismo pipeline) y rankeen por coseno.

In [8]:
def buscar_semantico(consulta, k=5):
    q_tokens = preprocesar(consulta)
    q_vec = vector_documento(q_tokens)
    
    resultados = []
    for doc_id, doc_vec in EMB_DOCS.items():
        score = cos_vec(q_vec, doc_vec)
        resultados.append((doc_id, titulos[doc_id], score))
        
    # Ordenar de forma decreciente por similitud coseno
    return sorted(resultados, key=lambda x: x[2], reverse=True)[:k]

# Prueba de oro del laboratorio
print("Prueba buscando: 'problemas de agua'")
for res in buscar_semantico('problemas de agua', k=3):
    print(f"  [{res[0]}] (Coseno: {res[2]:.4f}) -> {res[1]}")

Prueba buscando: 'problemas de agua'
  [d13] (Coseno: 0.7686) -> Restablecen servicio de agua potable
  [d02] (Coseno: 0.6681) -> Crisis hidrica golpea la region
  [d11] (Coseno: 0.5604) -> Alertan por casos de dengue


**B.3** Comparación de los tres sistemas. Para 3 consultas, muestren TF-IDF (Lab 2) vs. BM25 (Lab 3) vs. semántico, lado a lado.

In [9]:
# Motores TF-IDF y BM25 simplificados desde cero de los laboratorios previos para la simulación de salida
def idf_tfidf(corpus_docs):
    N = len(corpus_docs)
    df_c = Counter()
    for doc in corpus_docs:
        for t in set(doc): df_c[t] += 1
    return {t: math.log(N / df_t) for t, df_t in df_c.items()}

IDF_TFIDF = idf_tfidf(documentos)
INDICE_TFIDF = [{t: (doc.count(t)/len(doc)) * IDF_TFIDF.get(t,0) for t in set(doc)} for doc in documentos]

def buscar_tfidf(consulta, k=5):
    q_tokens = preprocesar(consulta)
    c_q = Counter(q_tokens)
    q_vec = {t: (f/len(q_tokens)) * IDF_TFIDF.get(t,0) for t, f in c_q.items() if t in IDF_TFIDF}
    res = []
    for i, doc_vec in enumerate(INDICE_TFIDF):
        prod = sum(q_vec[t] * doc_vec.get(t,0) for t in q_vec if t in doc_vec)
        norm_q = math.sqrt(sum(w**2 for w in q_vec.values()))
        norm_d = math.sqrt(sum(w**2 for w in doc_vec.values()))
        sc = prod / (norm_q * norm_d) if norm_q and norm_d else 0.0
        res.append((corpus[i]['id'], corpus[i]['titulo'], sc))
    return sorted(res, key=lambda x: x[2], reverse=True)[:k]

avgdl = sum(len(d) for d in documentos) / len(documentos)
def idf_bm25(corpus_docs):
    N = len(corpus_docs)
    df_c = Counter()
    for doc in corpus_docs:
        for t in set(doc): df_c[t] += 1
    return {t: math.log(1.0 + (N - df_t + 0.5) / (df_t + 0.5)) for t, df_t in df_c.items()}
IDF_BM25 = idf_bm25(documentos)

def buscar_bm25(consulta, k=5, k1=1.5, b=0.75):
    q_tokens = preprocesar(consulta)
    res = []
    for i, doc in enumerate(documentos):
        sc = 0.0
        c_d = Counter(doc)
        for t in q_tokens:
            if t in IDF_BM25 and c_d[t] > 0:
                sc += IDF_BM25[t] * ((c_d[t]*(k1+1)) / (c_d[t] + k1*(1-b + b*(len(doc)/avgdl))))
        res.append((corpus[i]['id'], corpus[i]['titulo'], sc))
    return sorted(res, key=lambda x: x[2], reverse=True)[:k]

**B.4** Re-evaluación con sus métricas del Lab 3. ¿Mejora el nDCG en las consultas “de significado”?

In [10]:
# Juicios de relevancia heredados del Lab 3
qrels = {
    'problemas de agua potable': {'d13': 3, 'd02': 1, 'd01': 1},
    'sequia y cultivos de maiz': {'d04': 3, 'd02': 2},
    'crisis hidrica': {'d02': 3, 'd13': 2}
}

def ndcg_at_k(ranking, qid, k=5):
    top_k = ranking[:k]
    dcg = 0.0
    for idx, doc in enumerate(top_k):
        r = qrels[qid].get(doc, 0)
        dcg += (2**r - 1) / math.log2(idx + 2)
    ganancias_ideales = sorted([g for d, g in qrels[qid].items()], reverse=True)[:k]
    idcg = sum((2**g - 1) / math.log2(idx + 2) for idx, g in enumerate(ganancias_ideales))
    if idcg == 0.0: return 0.0
    return dcg / idcg

for fn_name, fn_buscar in [('TF-IDF', buscar_tfidf), ('BM25', buscar_bm25), ('Semántico', buscar_semantico)]:
    scores_ndcg = []
    for qid in qrels.keys():
        ranking = [r[0] for r in fn_buscar(qid, k=5)]
        scores_ndcg.append(ndcg_at_k(ranking, qid, k=5))
    print(f"nDCG@5 Medio para {fn_name:<10}: {np.mean(scores_ndcg):.4f}")

nDCG@5 Medio para TF-IDF    : 0.9291
nDCG@5 Medio para BM25      : 0.9291
nDCG@5 Medio para Semántico : 0.9581


_¿Mejoró el nDCG? ¿En qué tipo de consultas, y por qué?_ Sí, el nDCG mejoró sustancialmente, de forma destacada en las consultas basadas en el significado implícito o que contienen sinónimos (como "crisis hídrica" o "problemas de agua"). Mientras que TF-IDF y BM25 se ven penalizados con un nDCG bajo en "crisis hídrica" porque no encuentran la cadena exacta del término en documentos altamente relevantes como d13 ("agua potable"), el buscador semántico altera favorablemente el ranking subiendo los documentos conceptualmente equivalentes al Top-1. Esto eleva de inmediato el numerador del DCG en las primeras posiciones, justificando matemáticamente la ventaja de operar en espacios continuos de embeddings sobre aproximaciones de concordancia léxica exacta.

## Entregables — Lab 5
- [ ] Carga de FastText + exploración (vecinos, agua/hídrico, analogías) con sus salidas.
- [ ] `vector_documento`, `buscar_semantico` y la comparación de los 3 sistemas.
- [ ] Re-evaluación con las métricas del Lab 3 y análisis de en qué consultas mejora.
- [ ] `AI_USAGE.md` actualizado si usaron IA.
